# SHARP-LLM: CodeT5-Base Convergence Experiment
## Kaggle T4 Notebook — 5 Epochs, Seed 50

Self-contained: clones the repo, installs requirements, copies data, and runs the experiment.
**Before running:** Add `msbasanth/sharp-llm-processed-data` via *Add Data → Your Datasets*.

In [ ]:
import subprocess
import sys
import os
import shutil
from pathlib import Path

# STEP 1: Move to safe directory FIRST
os.chdir('/kaggle/working')

# STEP 2: Remove old repo if it exists (ensures clean state on re-run)
repo_path = '/kaggle/working/repo'
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

# STEP 3: Install base dependencies before any SSL calls
print("Installing dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'transformers', 'scikit-learn', 'tqdm',
                'sentencepiece', 'truststore'],
               check=False)

# STEP 4: Clone fresh repository
print("Cloning repository...")
subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/msbasanth/sharp-llm.git', repo_path], check=True)

# STEP 4.5: Install full requirements from repo (keeps deps in sync)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', os.path.join(repo_path, 'requirements.txt')],
               check=False)

# STEP 5: Copy processed data from Kaggle dataset input → repo data/processed/
data_src = Path('/kaggle/input/sharp-llm-processed-data')
data_dst = Path(repo_path) / 'data' / 'processed'
data_dst.mkdir(parents=True, exist_ok=True)
for fname in ['train.parquet', 'test.parquet', 'label_map.json']:
    src = data_src / fname
    dst = data_dst / fname
    if not dst.exists():
        if src.exists():
            shutil.copy(src, dst)
            print(f"✓ Copied {fname}")
        else:
            raise FileNotFoundError(
                f"{src} not found — add 'msbasanth/sharp-llm-processed-data' as a dataset input"
            )
    else:
        print(f"✓ {fname} already present")

# STEP 6: Verify script exists
script_path = os.path.join(repo_path, 'scripts/convergence_experiment_codet5_base.py')
if not os.path.exists(script_path):
    raise FileNotFoundError(f"Script not found: {script_path}")

# STEP 7: Run convergence experiment
os.chdir(repo_path)
print("\nStarting CodeT5-Base convergence experiment...")
cmd = [sys.executable, 'scripts/convergence_experiment_codet5_base.py',
       '--model', 'Salesforce/codet5-base',
       '--epochs', '5',
       '--seed', '50',
       '--batch-size', '8',
       '--learning-rate', '2e-5',
       '--output-dir', '/kaggle/working/outputs']
subprocess.run(cmd, check=True)
print("✓ Convergence experiment complete!")

In [ ]:
import json
from pathlib import Path

output_dir = Path('/kaggle/working/outputs')
metrics_file = output_dir / 'epoch_metrics.json'

if metrics_file.exists():
    with open(metrics_file) as f:
        metrics = json.load(f)

    print("=" * 75)
    print("CONVERGENCE RESULTS — CodeT5-Base (5 Epochs, Seed 50)")
    print("=" * 75)
    print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train F1':>8}  {'Val F1':>6}  {'Test F1':>7}  {'Test Acc':>8}  {'MCC':>6}")
    print("-" * 75)
    for m in metrics:
        print(f"{m['epoch']:>5}  {m['train_loss']:>10.4f}  {m['train_f1']:>8.4f}  "
              f"{m['val_f1']:>6.4f}  {m['test_f1']:>7.4f}  {m['test_accuracy']:>8.4f}  {m['test_mcc']:>6.4f}")

    final = metrics[-1]
    print(f"\nFinal Test F1:  {final['test_f1']:.4f}")
    print(f"Final Accuracy: {final['test_accuracy']:.4f}")
    print(f"Final MCC:      {final['test_mcc']:.4f}")

    # Print JSON for log recovery (readable even if output files are lost)
    print("\nEPOCH_METRICS_JSON:", json.dumps(metrics))
else:
    print("epoch_metrics.json not found — check training output above for errors")

In [ ]:
# List all output files available for download
print("Output files:")
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(output_dir)}  ({f.stat().st_size // 1024} KB)")